# 08: 異なるモデル間のHidden State比較

## 目的
Notebook 07で「同一モデル3コピーでは構造が見えない」と判明。
異なるモデル（Qwen3-0.6B vs Qwen3-4B）のhidden stateに構造的差異があるか確認する。

## データ
llm-hidden-state-platformの既存実験データ（108実験中のphase0_baseline）を使用。
- Qwen3-0.6B: 100行 × 1024次元 × 6プロンプト
- Qwen3-4B: 100行 × 2560次元 × 6プロンプト
- 同一プロンプト・同一パラメータ（n_trials=10, temp=0.7, layer=-1）

## 判断基準
| 問い | 指標 | 閾値 |
|------|------|------|
| Q3: モデル間分離 | PCA共通空間でのsilhouette | > 0.1 |
| Q4: プロンプト依存性 | プロンプト別のモデル間距離比較 | 有意差あり |

## 課題: hidden_dimが異なる（1024 vs 2560）
→ PCAで共通次元に削減してから比較

In [ ]:
# === Cell 1: セットアップ ===
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('thoughtcomm'):
        !git clone https://github.com/AUMEZAK/thoughtcomm.git
    %cd thoughtcomm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# データはリポジトリ内に同梱（gzip圧縮）
DATA_DIR = 'data/platform_baseline/'

PROMPTS = {
    'P1': '1+1', 'P2': 'Japan capital', 'P3': 'Photosynthesis',
    'P4': 'Cat vs Dog', 'P5': 'Haiku', 'P6': 'Quantum'
}
MODELS = {'qwen3-0.6b': 1024, 'qwen3-4b': 2560}
DROP_COLS = ['trial_id', 'segment_index', 'segment_position']

print(f'Data dir: {DATA_DIR}')
print(f'Exists: {os.path.exists(DATA_DIR)}')

In [ ]:
# === Cell 2: データ読み込み ===
data = {}  # {(prompt_id, model_id): DataFrame}

for pid in PROMPTS:
    for mid in MODELS:
        csv_path = os.path.join(DATA_DIR, f'{pid}_{mid}', 'segments.csv.gz')
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path, compression='gzip')
            data[(pid, mid)] = df
            print(f'{pid}_{mid}: {df.shape}')
        else:
            print(f'{pid}_{mid}: NOT FOUND at {csv_path}')

print(f'\nLoaded: {len(data)} datasets')

In [ ]:
# === Cell 3: モデル間比較 — PCA共通空間 ===
# hidden_dimが異なるため、各モデルのデータを個別にPCAで共通次元に削減
# 方法: 各モデルのデータをPCA→64次元に削減、その後結合してさらにPCA→2D

N_COMMON = 64  # 共通次元数

# Step 1: 各モデルの全プロンプトデータを結合
model_data = {}
for mid in MODELS:
    frames = []
    prompt_labels = []
    for pid in PROMPTS:
        if (pid, mid) in data:
            df = data[(pid, mid)].drop(columns=DROP_COLS)
            frames.append(df.values)
            prompt_labels.extend([pid] * len(df))
    if frames:
        X = np.vstack(frames)
        model_data[mid] = {'X': X, 'prompts': np.array(prompt_labels)}
        print(f'{mid}: {X.shape} ({len(set(prompt_labels))} prompts)')

# Step 2: 各モデルを個別にPCA→N_COMMON次元
pca_models = {}
X_reduced = {}
for mid, d in model_data.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(d['X'])
    pca = PCA(n_components=N_COMMON)
    X_r = pca.fit_transform(X_scaled)
    pca_models[mid] = pca
    X_reduced[mid] = X_r
    print(f'{mid}: PCA {d["X"].shape[1]}d → {N_COMMON}d, '
          f'explained={pca.explained_variance_ratio_.sum():.1%}')

# Step 3: 結合して2D PCA
model_ids_list = list(MODELS.keys())
X_all = np.vstack([X_reduced[mid] for mid in model_ids_list])
model_labels = np.concatenate([
    np.full(len(X_reduced[mid]), mid) for mid in model_ids_list
])
prompt_labels_all = np.concatenate([
    model_data[mid]['prompts'] for mid in model_ids_list
])

pca_2d = PCA(n_components=3)
X_2d = pca_2d.fit_transform(X_all)
var_ratio = pca_2d.explained_variance_ratio_
print(f'\nCombined PCA: {X_all.shape} → 3D, '
      f'PC1={var_ratio[0]:.1%}, PC2={var_ratio[1]:.1%}, PC3={var_ratio[2]:.1%}')

In [ ]:
# === Cell 4: 散布図 — Q3(モデル間分離) ===
colors_model = {'qwen3-0.6b': 'tab:blue', 'qwen3-4b': 'tab:orange'}
colors_prompt = {'P1': 'tab:blue', 'P2': 'tab:green', 'P3': 'tab:red',
                 'P4': 'tab:purple', 'P5': 'tab:brown', 'P6': 'tab:pink'}

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# (A) Model color
for mid in model_ids_list:
    mask = model_labels == mid
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors_model[mid],
                    alpha=0.4, s=20, label=mid)
axes[0].set_title('(A) Model color — Q3: separate?')
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[0].legend()

# (B) Prompt color
for pid in PROMPTS:
    mask = prompt_labels_all == pid
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors_prompt[pid],
                    alpha=0.4, s=20, label=f'{pid}: {PROMPTS[pid]}')
axes[1].set_title('(B) Prompt color')
axes[1].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[1].legend(fontsize=8)

# (C) Model × Prompt
markers = {'qwen3-0.6b': 'o', 'qwen3-4b': 's'}
for mid in model_ids_list:
    for pid in PROMPTS:
        mask = (model_labels == mid) & (prompt_labels_all == pid)
        if mask.sum() > 0:
            axes[2].scatter(X_2d[mask, 0], X_2d[mask, 1],
                            c=colors_prompt[pid], marker=markers[mid],
                            alpha=0.4, s=25, label=f'{mid}/{pid}')
axes[2].set_title('(C) Model x Prompt (o=0.6B, s=4B)')
axes[2].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[2].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
# Legend too crowded, skip

plt.tight_layout()
os.makedirs('results/multimodel', exist_ok=True)
plt.savefig('results/multimodel/pca_model_prompt_scatter.png', dpi=150)
plt.show()

# PC1-PC3, PC2-PC3
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for mid in model_ids_list:
    mask = model_labels == mid
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 2], c=colors_model[mid],
                    alpha=0.4, s=20, label=mid)
    axes[1].scatter(X_2d[mask, 1], X_2d[mask, 2], c=colors_model[mid],
                    alpha=0.4, s=20, label=mid)
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[0].set_ylabel(f'PC3 ({var_ratio[2]:.1%})')
axes[0].set_title('Model color: PC1 vs PC3')
axes[0].legend()
axes[1].set_xlabel(f'PC2 ({var_ratio[1]:.1%})')
axes[1].set_ylabel(f'PC3 ({var_ratio[2]:.1%})')
axes[1].set_title('Model color: PC2 vs PC3')
axes[1].legend()
plt.tight_layout()
plt.savefig('results/multimodel/pca_model_pc13_pc23.png', dpi=150)
plt.show()

In [ ]:
# === Cell 5: 定量評価 ===
# Q3: モデル間分離
model_numeric = np.array([0 if m == 'qwen3-0.6b' else 1 for m in model_labels])
sil_model = silhouette_score(X_2d, model_numeric)

# プロンプト別分離
prompt_numeric = np.array([list(PROMPTS.keys()).index(p) for p in prompt_labels_all])
sil_prompt = silhouette_score(X_2d, prompt_numeric)

# モデル×プロンプトの12グループ
combined_labels = np.array([f'{m}_{p}' for m, p in zip(model_labels, prompt_labels_all)])
sil_combined = silhouette_score(X_2d, combined_labels)

print('=== Q3: Model Separation ===')
print(f'Silhouette (model):           {sil_model:.4f}')
print(f'Silhouette (prompt):          {sil_prompt:.4f}')
print(f'Silhouette (model x prompt):  {sil_combined:.4f}')

# Q4: プロンプト別のモデル間距離
print(f'\n=== Q4: Per-prompt model distance ===')
for pid in PROMPTS:
    mask_06 = (model_labels == 'qwen3-0.6b') & (prompt_labels_all == pid)
    mask_4b = (model_labels == 'qwen3-4b') & (prompt_labels_all == pid)
    if mask_06.sum() > 0 and mask_4b.sum() > 0:
        center_06 = X_2d[mask_06].mean(axis=0)
        center_4b = X_2d[mask_4b].mean(axis=0)
        inter_dist = np.linalg.norm(center_06 - center_4b)
        intra_06 = np.mean(np.linalg.norm(X_2d[mask_06] - center_06, axis=1))
        intra_4b = np.mean(np.linalg.norm(X_2d[mask_4b] - center_4b, axis=1))
        ratio = inter_dist / (0.5 * (intra_06 + intra_4b) + 1e-8)
        print(f'  {pid} ({PROMPTS[pid]:15s}): '
              f'inter={inter_dist:.2f}, intra_avg={0.5*(intra_06+intra_4b):.2f}, '
              f'ratio={ratio:.2f}')

In [ ]:
# === Cell 6: プロンプト別の詳細比較 ===
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i, pid in enumerate(PROMPTS):
    ax = axes[i // 3][i % 3]
    for mid in model_ids_list:
        mask = (model_labels == mid) & (prompt_labels_all == pid)
        if mask.sum() > 0:
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors_model[mid],
                       alpha=0.5, s=30, label=mid)
    ax.set_title(f'{pid}: {PROMPTS[pid]}')
    ax.legend(fontsize=8)
    ax.set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
    ax.set_ylabel(f'PC2 ({var_ratio[1]:.1%})')

plt.suptitle('Per-prompt: Qwen3-0.6B vs Qwen3-4B', fontsize=14)
plt.tight_layout()
plt.savefig('results/multimodel/pca_per_prompt.png', dpi=150)
plt.show()

In [ ]:
# === Cell 7: toorPIA比較（モデル間） ===
# PCA64d に削減したデータでtoorPIA basemap + addplot

!pip install -q git+https://github.com/toorpia/toorpia.git
from toorpia import toorPIA
import time

TOORPIA_API_KEY = 'toorpia_8acfd4e501d10123bd18bf5398051394c37cb08b0546b4a0'
client = toorPIA(api_key=TOORPIA_API_KEY)

# basemap: Qwen3-0.6B の全データ (PCA64d)
X_06 = X_reduced['qwen3-0.6b']
prompts_06 = model_data['qwen3-0.6b']['prompts']
df_base = pd.DataFrame(X_06, columns=[f'pc_{i}' for i in range(N_COMMON)])
df_base['model_id'] = 'qwen3-0.6b'
df_base['prompt_id'] = prompts_06
csv_base = 'results/multimodel/basemap_0.6b_pca64d.csv'
df_base.to_csv(csv_base, index=False)

print('Creating basemap: Qwen3-0.6B (PCA64d)')
result_base = client.basemap_csvform(
    csv_base,
    drop_columns=['model_id', 'prompt_id'],
    label='Qwen3-0.6B hidden states PCA64d (all prompts)',
    tag='multimodel-comparison',
)
print(f'Basemap: mapNo={result_base["mapNo"]}, {result_base["shareUrl"]}')
time.sleep(2)

# addplot: Qwen3-4B の全データ (PCA64d)
X_4b = X_reduced['qwen3-4b']
prompts_4b = model_data['qwen3-4b']['prompts']
df_add = pd.DataFrame(X_4b, columns=[f'pc_{i}' for i in range(N_COMMON)])
df_add['model_id'] = 'qwen3-4b'
df_add['prompt_id'] = prompts_4b
csv_add = 'results/multimodel/addplot_4b_pca64d.csv'
df_add.to_csv(csv_add, index=False)

print('Addplot: Qwen3-4B onto Qwen3-0.6B basemap')
result_add = client.addplot_csvform(
    csv_add,
    mapNo=result_base['mapNo'],
)
print(f'Addplot: abnormalityScore={result_add.get("abnormalityScore", "N/A")}, '
      f'status={result_add.get("abnormalityStatus", "N/A")}')
print(f'  {result_add.get("shareUrl", "N/A")}')

In [ ]:
# === Cell 8: toorPIA可視化 ===
xy_base = result_base['xyData']  # (600, 2) = 6 prompts x 100
# addplotのxyDataは返らないので、basemapのみで可視化

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (A) toorPIA: prompt color (basemap only)
for pid in PROMPTS:
    mask = prompts_06 == pid
    axes[0].scatter(xy_base[mask, 0], xy_base[mask, 1],
                    c=colors_prompt[pid], alpha=0.5, s=25,
                    label=f'{pid}: {PROMPTS[pid]}')
axes[0].set_title('toorPIA Basemap (Qwen3-0.6B): Prompt color')
axes[0].legend(fontsize=8)

# (B) trial color within one prompt
pid_focus = 'P3'
mask_p3 = prompts_06 == pid_focus
df_p3 = data[(pid_focus, 'qwen3-0.6b')]
trial_ids = df_p3['trial_id'].values
scatter = axes[1].scatter(xy_base[mask_p3, 0], xy_base[mask_p3, 1],
                          c=trial_ids, cmap='tab10', alpha=0.6, s=30)
axes[1].set_title(f'toorPIA: {pid_focus} trial color (Qwen3-0.6B)')
plt.colorbar(scatter, ax=axes[1], label='trial_id')

plt.tight_layout()
plt.savefig('results/multimodel/toorpia_basemap_scatter.png', dpi=150)
plt.show()

In [ ]:
# === Cell 9: 全結果サマリー ===
import json

print('=' * 70)
print('Notebook 08: Multi-model Comparison Results')
print('=' * 70)

print(f'\n=== PCA Analysis ===')
print(f'Common dim: {N_COMMON}')
print(f'Silhouette (model):          {sil_model:.4f}  {"YES" if sil_model > 0.1 else "NO"}')
print(f'Silhouette (prompt):         {sil_prompt:.4f}  {"YES" if sil_prompt > 0.1 else "NO"}')
print(f'Silhouette (model x prompt): {sil_combined:.4f}')

print(f'\n=== toorPIA ===')
print(f'Basemap (0.6B): mapNo={result_base["mapNo"]}')
print(f'  {result_base["shareUrl"]}')
print(f'Addplot (4B):   abnormalityScore={result_add.get("abnormalityScore", "N/A")}')
print(f'  {result_add.get("shareUrl", "N/A")}')

print(f'\n=== Verdict ===')
if sil_model > 0.1:
    verdict = 'Models are distinguishable — multi-model debate can create real private thoughts'
elif sil_prompt > 0.1:
    verdict = 'Prompts separate but models do not — task-level structure exists but model identity is weak'
else:
    verdict = 'Neither model nor prompt separation in PCA space'
print(f'  {verdict}')

summary = {
    'sil_model': float(sil_model),
    'sil_prompt': float(sil_prompt),
    'sil_combined': float(sil_combined),
    'toorpia_basemap_mapNo': result_base['mapNo'],
    'toorpia_basemap_url': result_base['shareUrl'],
    'toorpia_addplot_score': result_add.get('abnormalityScore'),
    'toorpia_addplot_status': result_add.get('abnormalityStatus'),
    'toorpia_addplot_url': result_add.get('shareUrl'),
    'verdict': verdict,
}
with open('results/multimodel/08_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: results/multimodel/08_summary.json')